# Roadtrip Map — Colab Build Pipeline

This notebook processes your Google Takeout ZIPs from Drive and produces:
- **`data.json`** — stop metadata, GPS clusters, route
- **`photos.zip`** — compressed display photos (~200KB each)

**Before running:**
1. Open the shared Drive link with the Takeout ZIPs
2. Right-click the folder → **Organize** → **Add shortcut** → **My Drive** → OK
3. Note the path (default: `My Drive/Takeout`)
4. Edit the `CONFIG` cell below, then **Runtime → Run all**

**After running:**
1. Download `data.json` → put it in `output/data.json`
2. Download `photos.zip` → extract into `output/photos/`
3. Refresh your local server

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Show top-level My Drive folders to help you verify the shortcut path
import os
print('\nTop-level folders in My Drive:')
try:
    for item in sorted(os.listdir('/content/drive/MyDrive')):
        if os.path.isdir(f'/content/drive/MyDrive/{item}'):
            n_files = len(os.listdir(f'/content/drive/MyDrive/{item}'))
            print(f'  📁 {item}  ({n_files} items)')
except Exception as e:
    print(f'Error listing Drive: {e}')
print('\n→ Find the folder containing your Takeout ZIPs above, then set TAKEOUT_PATH in the next cell.')

In [ ]:
%%bash
pip install Pillow pillow-heif requests -q

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIG  ←  edit these
# ─────────────────────────────────────────────────────────────────────────────

# Path to the folder containing your Takeout ZIPs.
# After mounting Drive above, set this to the path of the shortcut you added.
# Example: if the folder shortcut appears as "📁 Takeout" → keep as-is.
#          if it appears as "📁 GooglePhotos" → change to '/content/drive/MyDrive/GooglePhotos'
# Leave as None to auto-scan your whole Drive (slower, ~1 min).
TAKEOUT_PATH = '/content/drive/MyDrive/Takeout'   # ← check folder name above

# Trip info shown on the landing screen
TRIP_TITLE    = 'Southwest Odyssey'
TRIP_SUBTITLE = 'Nevada · Utah · Arizona · 2025'

# Clustering thresholds
OVERNIGHT_HOURS     = 4      # gap > this → overnight stop
DAY_MINUTES         = 30     # gap 30min–4h → day stop; < 30min → waypoint
CLUSTER_GAP_HOURS   = 3      # time gap that starts a new cluster
CLUSTER_RADIUS_KM   = 2.5    # distance gap that starts a new cluster

# Photo output settings
MAX_PHOTOS_PER_STOP = 12     # max photos shown in the carousel per stop
MAX_IMAGE_DIM       = 1200   # resize longest edge to this (pixels)
JPEG_QUALITY        = 82     # JPEG quality 1–95
GEOCODE_RATE_LIMIT  = 1.1    # seconds between Nominatim calls (don't lower this)

# ─── output paths (leave as-is) ──────────────────────────────────────────────
import os
OUTPUT_PHOTOS_DIR = '/content/roadtrip_photos'
DATA_JSON_PATH    = '/content/data.json'
os.makedirs(OUTPUT_PHOTOS_DIR, exist_ok=True)
print('Config OK.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS  (ported from build.py — do not edit)
# ─────────────────────────────────────────────────────────────────────────────
import json, math, io, zipfile, time, shutil
from datetime import datetime, timezone
from pathlib import Path
from collections import defaultdict
import requests
from PIL import Image, ImageOps

try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
    HEIC_SUPPORT = True
    print('HEIC support enabled.')
except Exception:
    HEIC_SUPPORT = False
    print('pillow-heif not available — HEIC files will be skipped for display.')

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.heic', '.heif', '.webp', '.gif', '.tif', '.tiff'}


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def centroid(photos):
    lats = [p['lat'] for p in photos if p.get('lat')]
    lons = [p['lon'] for p in photos if p.get('lon')]
    if not lats:
        return None, None
    return sum(lats)/len(lats), sum(lons)/len(lons)


def parse_sidecar_data(data, photo_name):
    geo = data.get('geoData') or {}
    lat, lon = geo.get('latitude', 0.0), geo.get('longitude', 0.0)
    if lat == 0.0 and lon == 0.0:
        geo = data.get('geoDataExif') or {}
        lat, lon = geo.get('latitude', 0.0), geo.get('longitude', 0.0)
    has_gps = not (lat == 0.0 and lon == 0.0)
    ts_raw = (data.get('photoTakenTime') or data.get('creationTime') or {}).get('timestamp', '0')
    try:
        ts = datetime.fromtimestamp(int(ts_raw), tz=timezone.utc)
    except Exception:
        ts = None
    return {
        'filename':    photo_name,
        'timestamp':   ts,
        'lat':         lat if has_gps else None,
        'lon':         lon if has_gps else None,
        'has_gps':     has_gps,
        'description': data.get('description', ''),
        'title':       data.get('title', ''),
    }


def deduplicate(photos):
    seen, unique = set(), []
    for p in photos:
        if not p['timestamp']:
            unique.append(p)
            continue
        key = (int(p['timestamp'].timestamp()//5),
               round(p['lat'] or 0, 4),
               round(p['lon'] or 0, 4))
        if key not in seen:
            seen.add(key)
            unique.append(p)
    return unique


def cluster_photos(gps_photos):
    if not gps_photos:
        return []
    photos = sorted(gps_photos, key=lambda p: p['timestamp'])
    clusters, current = [], [photos[0]]
    for photo in photos[1:]:
        prev = current[-1]
        gap_h = (photo['timestamp'] - prev['timestamp']).total_seconds()/3600 \
                if prev['timestamp'] and photo['timestamp'] else 0
        clat, clon = centroid(current)
        dist = haversine_km(clat, clon, photo['lat'], photo['lon']) \
               if clat and photo['lat'] else 0
        if gap_h > CLUSTER_GAP_HOURS or dist > CLUSTER_RADIUS_KM:
            clusters.append(current)
            current = [photo]
        else:
            current.append(photo)
    clusters.append(current)
    return clusters


def classify_stop(cluster):
    ts = [p['timestamp'] for p in cluster if p['timestamp']]
    if len(ts) < 2:
        return 'waypoint'
    dur_h = (max(ts) - min(ts)).total_seconds() / 3600
    if dur_h >= OVERNIGHT_HOURS:
        return 'overnight'
    if dur_h >= DAY_MINUTES / 60:
        return 'day'
    return 'waypoint'


def select_representative(cluster):
    clat, clon = centroid(cluster)
    if not clat:
        return cluster[0]
    gps = [p for p in cluster if p['lat']]
    return min(gps or cluster,
               key=lambda p: haversine_km(clat, clon, p['lat'] or 0, p['lon'] or 0))


_last_geo = 0.0
def reverse_geocode(lat, lon):
    global _last_geo
    elapsed = time.time() - _last_geo
    if elapsed < GEOCODE_RATE_LIMIT:
        time.sleep(GEOCODE_RATE_LIMIT - elapsed)
    try:
        r = requests.get(
            'https://nominatim.openstreetmap.org/reverse',
            params={'lat': lat, 'lon': lon, 'format': 'json', 'zoom': 10},
            headers={'User-Agent': 'roadtrip-map/1.0 (personal project)'},
            timeout=10)
        _last_geo = time.time()
        addr = r.json().get('address', {})
        parts = [
            addr.get('tourism') or addr.get('leisure') or addr.get('natural'),
            addr.get('city') or addr.get('town') or addr.get('village') or addr.get('county'),
            addr.get('state'),
        ]
        return ', '.join(p for p in parts if p) or f'{lat:.4f}, {lon:.4f}'
    except Exception:
        _last_geo = time.time()
        return f'{lat:.4f}, {lon:.4f}'


def resize_image(img_bytes, ext):
    try:
        img = Image.open(io.BytesIO(img_bytes))
        img = ImageOps.exif_transpose(img)   # fix orientation
        if img.mode not in ('RGB', 'L'):
            img = img.convert('RGB')
        if max(img.width, img.height) > MAX_IMAGE_DIM:
            ratio = MAX_IMAGE_DIM / max(img.width, img.height)
            img = img.resize((int(img.width*ratio), int(img.height*ratio)), Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=JPEG_QUALITY, optimize=True)
        return buf.getvalue()
    except Exception as e:
        print(f'    Warning: could not resize image: {e}')
        return None


print('Helper functions loaded.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Discover and index ZIP files
# Reads only each ZIP's central directory (the index at the end of the file),
# NOT the full file contents.  Fast even for 10GB ZIPs.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess
from pathlib import Path

def _resolve_takeout_path(configured):
    """Return the directory containing Takeout ZIPs, auto-detecting if needed."""
    if configured:
        p = Path(configured)
        if p.exists() and list(p.glob('*.zip')):
            return p
        if p.exists():
            print(f'Warning: {configured} exists but has no .zip files.')
        else:
            print(f'Warning: {configured} not found in Drive.')

    print('Auto-scanning /content/drive/MyDrive for ZIP files (may take ~60s)...')
    result = subprocess.run(
        ['find', '/content/drive/MyDrive', '-name', '*.zip', '-maxdepth', '4'],
        capture_output=True, text=True, timeout=120
    )
    found = [p for p in result.stdout.strip().split('\n') if p.strip()]
    if not found:
        raise FileNotFoundError(
            'No .zip files found in My Drive (searched 4 levels deep).\n'
            'Make sure you added a shortcut to the shared Takeout folder:\n'
            '  Open the shared link → right-click → Organize → Add shortcut → My Drive'
        )
    # Group by parent directory, pick dir with most ZIPs
    dirs = {}
    for f in found:
        d = str(Path(f).parent)
        dirs[d] = dirs.get(d, 0) + 1
    best = max(dirs, key=dirs.get)
    print(f'Auto-detected: {best}  ({dirs[best]} ZIP files)\n'
          f'Tip: set TAKEOUT_PATH = {repr(best)} in the config cell to skip this scan next time.')
    return Path(best)

takeout_dir = _resolve_takeout_path(TAKEOUT_PATH)
print(f'\nUsing: {takeout_dir}')

zip_paths = sorted(takeout_dir.glob('*.zip'))
print(f'Found {len(zip_paths)} ZIP file(s):')
for z in zip_paths:
    size_gb = z.stat().st_size / 1e9
    print(f'  {z.name}  ({size_gb:.1f} GB)')

# Build two indices:
#   photo_index  : 'IMG_1234.jpg' → (zip_path, 'Takeout/Google Photos/.../IMG_1234.jpg')
#   sidecar_list : list of (zip_path, member_name, sidecar_basename)
import zipfile
photo_index  = {}   # filename → (zip_path, member_path)
sidecar_list = []   # (zip_path, member_path, basename)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.heic', '.heif', '.webp', '.gif', '.tif', '.tiff'}

for zip_path in zip_paths:
    print(f'  Indexing {zip_path.name} ...', end=' ', flush=True)
    n_photos = n_sidecars = 0
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name     = info.filename
            basename = Path(name).name
            ext      = Path(basename).suffix.lower()
            stem     = Path(basename).stem          # 'IMG.jpg.json' → 'IMG.jpg'
            stem_ext = Path(stem).suffix.lower()    # 'IMG.jpg' → '.jpg'

            if ext in IMAGE_EXTS:
                if basename not in photo_index:     # first ZIP wins on dupe filenames
                    photo_index[basename] = (zip_path, name)
                n_photos += 1
            elif ext == '.json' and stem_ext in IMAGE_EXTS:
                sidecar_list.append((zip_path, name, basename))
                n_sidecars += 1

    print(f'{n_photos} photos, {n_sidecars} sidecars')

print(f'\nTotal: {len(photo_index):,} photos indexed, {len(sidecar_list):,} sidecars to parse')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Extract GPS + timestamps from sidecar JSONs
# Only the tiny .json files are read; photos are not touched yet.
# ─────────────────────────────────────────────────────────────────────────────
print(f'Parsing {len(sidecar_list):,} sidecar files...')

all_photos = []

# Group by ZIP so we open each ZIP only once
sidecars_by_zip = defaultdict(list)
for zip_path, member, basename in sidecar_list:
    sidecars_by_zip[zip_path].append((member, basename))

for zip_path, entries in sidecars_by_zip.items():
    with zipfile.ZipFile(zip_path) as zf:
        for member, basename in entries:
            photo_name = Path(basename).stem   # 'IMG_1234.jpg.json' → 'IMG_1234.jpg'
            try:
                data   = json.loads(zf.read(member))
                parsed = parse_sidecar_data(data, photo_name)
                if parsed and parsed['timestamp']:
                    all_photos.append(parsed)
            except Exception:
                pass

print(f'Parsed {len(all_photos):,} photos with timestamps')
all_photos = deduplicate(all_photos)
print(f'After dedup: {len(all_photos):,} unique photos')

gps_photos  = [p for p in all_photos if p['has_gps']]
nogps_photos = [p for p in all_photos if not p['has_gps']]
print(f'  GPS: {len(gps_photos):,}   no-GPS: {len(nogps_photos):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Cluster into stops and reverse-geocode
# ─────────────────────────────────────────────────────────────────────────────
print('Clustering photos into stops...')
clusters = cluster_photos(gps_photos)
print(f'Found {len(clusters)} raw clusters')

# Attach no-GPS photos to nearest cluster by time
for p in nogps_photos:
    if not p['timestamp'] or not clusters:
        continue
    best = min(clusters, key=lambda c: min(
        abs((p['timestamp'] - q['timestamp']).total_seconds())
        for q in c if q['timestamp']
    ), default=None)
    if best:
        best.append(p)

stop_clusters     = [c for c in clusters if classify_stop(c) in ('overnight', 'day')]
waypoint_clusters = [c for c in clusters if classify_stop(c) == 'waypoint']
overnight_n = sum(1 for c in stop_clusters if classify_stop(c) == 'overnight')
day_n       = len(stop_clusters) - overnight_n
print(f'  {overnight_n} overnight stops, {day_n} day stops, {len(waypoint_clusters)} waypoints')

print(f'\nReverse-geocoding {len(stop_clusters)} stops (Nominatim, ~{len(stop_clusters)*1.2:.0f}s)...')
geocoded = []
for i, cluster in enumerate(stop_clusters):
    stop_type = classify_stop(cluster)
    clat, clon = centroid(cluster)
    if not clat:
        continue
    ts  = [p['timestamp'] for p in cluster if p['timestamp']]
    loc = reverse_geocode(clat, clon)
    geocoded.append({
        '_cluster':    cluster,
        '_type':       stop_type,
        '_lat':        clat,
        '_lon':        clon,
        '_arrival':    min(ts),
        '_departure':  max(ts),
        '_duration_h': (max(ts) - min(ts)).total_seconds()/3600 if len(ts) > 1 else 0,
        '_name':       loc.split(',')[0].strip(),
        '_location':   loc,
        '_rep':        select_representative(cluster),
    })
    print(f'  {i+1}/{len(stop_clusters)}: {loc.split(",")[0].strip()} ({stop_type})')

print(f'\nGeocoded {len(geocoded)} stops.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Extract and resize display photos from ZIPs
# Only fetches the specific bytes for the photos actually shown in the app.
# ─────────────────────────────────────────────────────────────────────────────
print(f'Extracting up to {MAX_PHOTOS_PER_STOP} display photos per stop...')

# Collect which photos we need
to_extract = {}   # original_filename → output_jpg_path
for s in geocoded:
    for p in s['_cluster'][:MAX_PHOTOS_PER_STOP]:
        fname = p['filename']
        if fname in photo_index:
            out_name = Path(fname).stem + '.jpg'
            to_extract[fname] = os.path.join(OUTPUT_PHOTOS_DIR, out_name)

print(f'Need {len(to_extract)} unique photos across all stops')

# Group by ZIP for efficient batch reads
by_zip = defaultdict(list)
for fname, out_path in to_extract.items():
    zip_path, member = photo_index[fname]
    by_zip[zip_path].append((fname, member, out_path))

total = len(to_extract)
done  = 0
errors = 0

for zip_path, entries in by_zip.items():
    print(f'  {zip_path.name}: {len(entries)} photos to extract')
    with zipfile.ZipFile(zip_path) as zf:
        for fname, member, out_path in entries:
            if os.path.exists(out_path):
                done += 1
                continue
            ext = Path(fname).suffix.lower()
            if ext in ('.heic', '.heif') and not HEIC_SUPPORT:
                done += 1
                continue
            try:
                img_bytes = zf.read(member)
                resized   = resize_image(img_bytes, ext)
                if resized:
                    with open(out_path, 'wb') as f:
                        f.write(resized)
            except Exception as e:
                print(f'    Error on {fname}: {e}')
                errors += 1
            done += 1
            if done % 20 == 0:
                print(f'    {done}/{total} photos processed...')

saved = len([f for f in os.listdir(OUTPUT_PHOTOS_DIR) if f.endswith('.jpg')])
total_mb = sum(os.path.getsize(os.path.join(OUTPUT_PHOTOS_DIR, f))
               for f in os.listdir(OUTPUT_PHOTOS_DIR)) / 1e6
print(f'\nDone: {saved} photos saved, {total_mb:.1f} MB total, {errors} errors')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Build data.json
# ─────────────────────────────────────────────────────────────────────────────

# Guard: make sure earlier steps ran in this session
assert 'geocoded' in dir() and geocoded, (
    'geocoded is empty — did Step 3 run in this session?\n'
    f'  gps_photos found: {len(gps_photos) if "gps_photos" in dir() else "unknown"}\n'
    f'  clusters found:   {len(clusters) if "clusters" in dir() else "unknown"}\n'
    'If the runtime reset, re-run from Step 1.'
)


def interpolate_segment(p1, p2, steps=20):
    return [[p1[0] + (p2[0]-p1[0])*i/steps,
             p1[1] + (p2[1]-p1[1])*i/steps] for i in range(steps)]


stops_out   = []
hero_photos = []
stop_order  = 0

for s in geocoded:
    stop_order += 1
    cluster  = s['_cluster']
    rep_name = s['_rep']['filename']
    rep_out  = Path(rep_name).stem + '.jpg'
    rep_url  = f'photos/{rep_out}'

    photos_out = []
    for p in cluster[:MAX_PHOTOS_PER_STOP]:
        fname    = p['filename']
        out_name = Path(fname).stem + '.jpg'
        out_path = os.path.join(OUTPUT_PHOTOS_DIR, out_name)
        if not os.path.exists(out_path):
            continue
        photos_out.append({
            'id':        fname,
            'filename':  out_name,
            'url':       f'photos/{out_name}',
            'timestamp': p['timestamp'].isoformat() if p['timestamp'] else None,
            'caption':   p['description'] or p['title'] or '',
        })

    stops_out.append({
        'id':                   f'stop_{stop_order:02d}',
        'name':                 s['_name'],
        'location':             s['_location'],
        'lat':                  round(s['_lat'], 6),
        'lng':                  round(s['_lon'], 6),
        'arrival':              s['_arrival'].isoformat(),
        'departure':            s['_departure'].isoformat(),
        'duration_hours':       round(s['_duration_h'], 1),
        'type':                 s['_type'],
        'order':                stop_order,
        'representative_photo': rep_url,
        'photos':               photos_out,
    })

    if s['_type'] == 'overnight' and len(hero_photos) < 3:
        hero_photos.append(rep_url)

# Route — interpolate between stops
pts = [[s['lng'], s['lat']] for s in stops_out]
coords = []
if len(pts) >= 2:
    for i in range(len(pts) - 1):
        coords.extend(interpolate_segment(pts[i], pts[i+1]))
    coords.append(pts[-1])
elif len(pts) == 1:
    coords = pts  # single stop — no route to draw

# Waypoints
waypoints_out = []
for i, cluster in enumerate(waypoint_clusters):
    clat, clon = centroid(cluster)
    if not clat:
        continue
    ts = [p['timestamp'] for p in cluster if p['timestamp']]
    waypoints_out.append({
        'id':        f'wp_{i:02d}',
        'lat':       round(clat, 6),
        'lng':       round(clon, 6),
        'timestamp': min(ts).isoformat() if ts else None,
        'name':      '',
    })

data = {
    'trip': {
        'title':       TRIP_TITLE,
        'subtitle':    TRIP_SUBTITLE,
        'start_date':  stops_out[0]['arrival'][:10]    if stops_out else '',
        'end_date':    stops_out[-1]['departure'][:10] if stops_out else '',
        'hero_photos': hero_photos,
    },
    'stops':     stops_out,
    'waypoints': waypoints_out,
    'route': {
        'type': 'Feature',
        'geometry': {'type': 'LineString', 'coordinates': coords},
    },
}

with open(DATA_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

total_display = sum(len(s['photos']) for s in stops_out)
overnight_n   = sum(1 for s in stops_out if s['type'] == 'overnight')
day_n         = sum(1 for s in stops_out if s['type'] == 'day')
print(f'data.json written.')
print(f'  {overnight_n} overnight + {day_n} day stops')
print(f'  {total_display} display photos across all stops')
print(f'  {len(waypoints_out)} waypoints')
print(f'  {len(coords)} route coordinates')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — Download results
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import files

print('Packaging photos.zip...')
shutil.make_archive('/content/roadtrip_photos_bundle', 'zip', OUTPUT_PHOTOS_DIR)
bundle_mb = os.path.getsize('/content/roadtrip_photos_bundle.zip') / 1e6
print(f'  photos.zip: {bundle_mb:.1f} MB')

print('Downloading data.json (~50KB)...')
files.download(DATA_JSON_PATH)

print(f'Downloading roadtrip_photos_bundle.zip ({bundle_mb:.0f} MB)...')
files.download('/content/roadtrip_photos_bundle.zip')

print()
print('--- Done! ---')
print('Next steps on your local machine:')
print('  1. Copy data.json to output/data.json')
print('  2. Extract roadtrip_photos_bundle.zip into output/photos/')
print('     (overwrite the SVG placeholders)')
print('  3. Refresh your browser at http://localhost:8080')